# CamusGPT v3 — Phase 1: Voice SFT on Gemma-4-12B

Trains the **voice** LoRA on `unsloth/gemma-4-12b-it`. Output: an adapter folder on Drive
whose name embeds `PHASE1_CORPUS`, so the `old` and `new` corpus runs can never overwrite
each other. This is the INPUT to the Phase-2 refusal-SFT notebook.

**Why run Phase 1 twice.** `docs/gemma4_assessment.md` §7 recommends switching base and
switching corpus together, but not blindly: if the persona improves you otherwise cannot
tell which change did it. Phase 1 is the cheap pass and the one that owns voice, so it is
run twice on Gemma 4 — once per corpus — and a single Phase 2 follows on the winner.
Set `PHASE1_CORPUS` below to choose the run; nothing else changes.

**Gemma-4 differences that matter here (vs the Gemma-3 run this is derived from):**
1. **Turn markers changed.** `<|turn>` / `<turn|>`, not the older Gemma markup. Response
   masking is retargeted, and CHECK 2 now checks masking *content*, not merely "something
   was unmasked".
2. **Native system role.** Gemma 4 keeps a system message as its own turn instead of
   folding it into the first user turn, so a row's leading system message is passed
   straight through. The old fold-into-user code is gone.
3. **Thinking.** Gemma 4 has a reasoning mode. Training renders with it disabled, and a
   cell asserts that no rendered target carries a non-empty thought channel.
4. **`FastModel`.** This is the loader the model card uses for this repo.
5. **`token_type_ids`.** Gemma 3's causal-mask builder demanded them. That may be
   architecture-specific, so it is no longer assumed: one real batch is pushed through
   the model and the zero-filled wrapper is installed only if the error names
   `token_type_ids`. The cell prints which path it took.


## Data: the shared `data_v3/` format

This notebook reads **one** pre-assembled file instead of the per-corpus files the Gemma-3
run loaded and mixed itself:

- `MyDrive/CamusGPT_Training/data_v3/phase1_new.jsonl`  (9,788 rows)
- `MyDrive/CamusGPT_Training/data_v3/phase1_old.jsonl`  (8,918 rows)

Selected by `PHASE1_CORPUS` below. Each row is:

```json
{"messages": [{"role": "...", "content": "..."}, ...], "kind": "...", "source": "..."}
```

`messages` may begin with a `role: "system"` message (kept as a real system turn on
Gemma 4). `kind` and `source` are provenance labels.

`build/training_v3/MANIFEST.json` (uploaded alongside the data, so it is in the same
folder) records for each set: row count, `kind` histogram, sha256, assistant-turn metrics,
and the exact per-input weights. The data cell reads it and checks the file it loaded
against it.

### Upweighting: the assembler does none for Phase 1, and the notebook must not invent any

`MANIFEST.json` shows **every Phase-1 input at weight 1** — there is no upweighting in
`phase1_old.jsonl` or `phase1_new.jsonl`. So:

- The Gemma-3 Phase-1 notebook's `rows = sft + conv * 3` is **gone**. Nothing replaces it.
- **`phase1_old` is therefore not a bit-exact replay of the camus2 Phase 1.** Same input
  files, same 8,918 unique rows, but 9,126 training rows back then (the 3x on the 104
  conversational rows) against 8,918 now. The conversational *share* barely moves
  (1.1% → 1.2%), so this is a small effect — but it is a second variable changing at the
  same time as the base, on top of the corpus change itself. If the two Phase-1 runs come
  out close, that is three candidate causes, not one.
- If you decide the conversational rows *should* still be tripled, add the weight in
  `pipeline/assemble_training.py` and re-run it, so the manifest records it. Do not
  multiply rows here: a notebook-side weight makes the delivered corpus and the trained
  corpus different objects, and nothing downstream can tell which one produced a result.

The data cell prints the delivered row count, the weight list, the sha256 check, and a
`kind`/`source` histogram so the mixture is visible rather than assumed.


In [ ]:
# 1) Environment
!pip install -q --no-deps unsloth vllm==0.* 2>/dev/null
!pip install -q unsloth
import torch, transformers, unsloth
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| unsloth", unsloth.__version__)
print("GPU:", torch.cuda.get_device_name(0),
      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB")


In [ ]:
# 2) Drive + config. T0 is the stopwatch behind the wall-clock figure.
import time
T0 = time.time()

from google.colab import drive
drive.mount('/content/drive')

DRIVE       = "/content/drive/MyDrive/CamusGPT_Training"
BASE_MODEL  = "unsloth/gemma-4-12b-it"
DATA_DIR    = f"{DRIVE}/data_v3"

# "new" = primary-text-augmented corpus, "old" = the corpus the Gemma-3 run used.
# The adapter folder name carries it, so the two runs never overwrite each other.
PHASE1_CORPUS = "new"
assert PHASE1_CORPUS in ("new", "old"), f"unexpected corpus: {PHASE1_CORPUS}"

SRC         = f"{DATA_DIR}/phase1_{PHASE1_CORPUS}.jsonl"
OUT_ADAPTER = f"{DRIVE}/adapters/camus3_voice_lora_{PHASE1_CORPUS}"
MAX_SEQ     = 2048

import os
os.makedirs(f"{DRIVE}/adapters", exist_ok=True)
assert os.path.exists(SRC), (
    f"MISSING: {SRC}\nUpload the data_v3 files to {DATA_DIR} first. Expected one of "
    "phase1_new.jsonl / phase1_old.jsonl.")
print("corpus:", SRC, os.path.getsize(SRC), "bytes")
print("adapter:", OUT_ADAPTER)


In [ ]:
# 3) Load Gemma-4-12B + attach a fresh LoRA
# FastModel (not FastLanguageModel) is what this repo's own card uses.
from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ,
    load_in_4bit = True,          # QLoRA: 12B fits comfortably on A100/L4
    dtype = None,                 # auto -> bf16 where supported
)
model = FastModel.get_peft_model(
    model,
    r = 32,                       # voice pass: a little more capacity than the guardrail pass
    lora_alpha = 32,
    lora_dropout = 0.0,
    bias = "none",
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print(model.print_trainable_parameters())


In [ ]:
# 4) Chat template — Gemma 4's own, thinking OFF, native system role kept
# The base repo's own template is the canonical one. Do NOT force another
# architecture's template here; that is how the wrong turn markers get baked in.

assert getattr(tokenizer, "chat_template", None), (
    "the loaded tokenizer carries no chat_template — the model card's template is "
    "missing. Check the base repo before going further; do not substitute another "
    "architecture's markup.")

# Unsloth may hand back a wrapper; the plain HF tokenizer underneath is the one whose
# decode() and convert_tokens_to_ids() speak Gemma 4's native markup.
tok = getattr(tokenizer, "tokenizer", tokenizer)

INSTRUCTION_PART = "<|turn>user\n"      # opens a user turn
RESPONSE_PART    = "<|turn>model\n"     # opens a model (assistant) turn
EOT_TOKEN        = "<turn|>"            # closes a turn

THOUGHT_OPEN  = "<|channel>thought\n"
THOUGHT_CLOSE = "<channel|>"

def thought_payloads(text):
    """Every thought-channel body in `text` ("" == empty, i.e. thinking disabled)."""
    out, i = [], 0
    while True:
        j = text.find(THOUGHT_OPEN, i)
        if j < 0:
            return out
        start = j + len(THOUGHT_OPEN)
        k = text.find(THOUGHT_CLOSE, start)
        if k < 0:                       # unterminated channel
            out.append(text[start:])
            return out
        out.append(text[start:k])
        i = k + len(THOUGHT_CLOSE)

def assert_no_thoughts(texts, label="rendered texts"):
    """Gemma 4 thinks unless told not to. A training target must never carry a thought."""
    bad = [(i, thought_payloads(t)) for i, t in enumerate(texts)
           if any(p.strip() for p in thought_payloads(t))]
    assert not bad, (
        f"{len(bad)} of {label} carry a NON-EMPTY thought channel; first offender "
        f"index {bad[0][0]}: {bad[0][1]!r}. apply_chat_template must be called with "
        "enable_thinking=False.")
    print(f"  no thought channel in {len(texts)} {label}")

def norm(s):
    return " ".join(s.split())

def render(msgs, add_generation_prompt=False):
    """The single rendering path for this notebook: text, thinking disabled."""
    return tokenizer.apply_chat_template(
        msgs, tokenize=False, add_generation_prompt=add_generation_prompt,
        enable_thinking=False)

def last_assistant(messages):
    a = [m["content"] for m in messages if m["role"] == "assistant"]
    return a[-1] if a else ""

probe = render([
    {"role": "system", "content": "SYS_HERE"},
    {"role": "user",   "content": "PROMPT_HERE"},
    {"role": "assistant", "content": "RESPONSE_HERE"},
])
print(repr(probe))
print(probe)

assert RESPONSE_PART in probe, "unexpected template: no model turn marker"
assert "<|turn>system\n" in probe, (
    "Gemma 4 has a native system role — a system message must stay its own turn, "
    "not be folded into the user turn.")
assert "<|turn>user\nSYS_HERE" not in probe, "system text leaked into the user turn"
assert "PROMPT_HERE" in probe and "RESPONSE_HERE" in probe
assert not any(p.strip() for p in thought_payloads(probe)), "thinking is not disabled"

EOT_ID = tok.convert_tokens_to_ids(EOT_TOKEN)
assert EOT_ID is not None and EOT_ID != tok.unk_token_id, (
    f"{EOT_TOKEN} is not a single token in this tokenizer (id={EOT_ID}) — masking and "
    "generation would both be wrong. Stop and check the tokenizer.")

print("\nCHECK 0 PASSED — gemma-4 template, system turn kept, thinking off")
print("EOT", EOT_TOKEN, "->", EOT_ID, "| eos:", tokenizer.eos_token_id)


In [ ]:
# 5) Load the shared-format corpus from data_v3, verified against its manifest
import json, random, hashlib, os
from collections import Counter

def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

# --- the manifest describes exactly what the assembler delivered ---------------
# It ships to Drive in the same folder, so the notebook can check the file it is about
# to train on against the file that was built.
MANIFEST = f"{DATA_DIR}/MANIFEST.json"
SET_NAME  = f"phase1_{PHASE1_CORPUS}"
mset, manifest_ok = None, False
if os.path.exists(MANIFEST):
    man = json.load(open(MANIFEST, encoding="utf-8"))
    mset = man["sets"][SET_NAME]
    weights = [g.get("weight") for g in mset.get("inputs", []) if g.get("weight") is not None]
    upweighted = sorted({w for w in weights if w != 1})
    print(f"manifest {SET_NAME}: {mset['rows']} rows, file={mset['file']}")
    print("  assembler weights:", weights or "none recorded",
          "-> upweighting already applied" if upweighted
          else "-> NO upweighting applied (every input weight 1); do not add any here")
    print("  note:", mset.get("note", "-"))
    print("  corpus tag:", mset.get("corpus_tag", "-"), "| exact dup turns dropped:",
          mset["metrics"].get("exact_duplicate_turns", "-"))
    sha = hashlib.sha256(open(SRC, "rb").read()).hexdigest()
    manifest_ok = (sha == mset.get("sha256"))
    print(f"  sha256 {'matches' if manifest_ok else 'DOES NOT MATCH'}: {sha[:16]}...")
    assert manifest_ok, (
        f"{SRC} is not the file MANIFEST.json describes.\n  manifest: {mset.get('sha256')}\n"
        f"  on Drive: {sha}\nIf you regenerated the corpus, re-upload MANIFEST.json with it; "
        "otherwise the two Phase-1 runs are not comparable and the A/B is meaningless.")
else:
    print("WARNING: no MANIFEST.json beside the corpus — proceeding unverified.")
    print("         The delivered row count and mixture below are all you have; record them.")

rows_raw = load_jsonl(SRC)
examples, dropped = [], 0
for r in rows_raw:
    msgs = r.get("messages") or []
    msgs = [{"role": m["role"], "content": m["content"]} for m in msgs
            if m.get("content") is not None]
    if not msgs or not any(m["role"] == "assistant" for m in msgs):
        dropped += 1
        continue
    examples.append({"messages": msgs,
                     "kind": r.get("kind", "?"),
                     "source": r.get("source", "?")})

print(f"\nread {len(rows_raw)} rows -> {len(examples)} usable ({dropped} dropped: "
      "no messages, or no assistant turn)")
print("kind  :", dict(Counter(e["kind"]   for e in examples).most_common()))
print("source:", dict(Counter(e["source"] for e in examples).most_common()))
n_sys = sum(1 for e in examples if e["messages"][0]["role"] == "system")
print(f"rows with a leading system message: {n_sys}/{len(examples)} (kept as a real "
      "system turn — Gemma 4 has a native system role)")

assert dropped == 0, (
    f"{dropped} rows had no usable assistant turn. The manifest promises every row ends "
    "with one, so either the file is not the assembled one or the schema changed.")
if manifest_ok:
    assert len(examples) == mset["rows"], (
        f"row count {len(examples)} != manifest {mset['rows']} on a file whose sha256 "
        "matches — the notebook's loader is dropping or duplicating rows. Stop.")
    print(f"row count matches the manifest ({len(examples)})")
else:
    assert len(examples) >= 1000, (
        f"corpus looks short: {len(examples)} usable rows. If that is genuinely correct "
        "for your data, lower this floor deliberately rather than by accident.")

random.Random(3407).shuffle(examples)
print("median final-assistant words:",
      sorted(len(last_assistant(e["messages"]).split()) for e in examples)[len(examples)//2])


In [ ]:
# 6) Render every row once (thinking off), then pre-flight
from datasets import Dataset

texts = [render(e["messages"]) for e in examples]
assert_no_thoughts(texts, "corpus rows")

ds = Dataset.from_list([{"text": t} for t in texts])
print(ds[0]["text"][:600])
print("...\nrows:", len(ds))

# ================================ CHECK 1 — data pre-flight ================================
assert RESPONSE_PART in ds[0]["text"]
assert not [i for i, t in enumerate(texts) if not norm(t)], "rows rendered empty"
lens = [len(tok(t)["input_ids"]) for t in texts[:400]]
print(f"sample token lengths: median={sorted(lens)[len(lens)//2]}, max={max(lens)}, "
      f"over MAX_SEQ={sum(l > MAX_SEQ for l in lens)}/400")
print("CHECK 1 PASSED")


In [ ]:
# 7) Trainer with response-only masking (Gemma 4 turn markers)
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds,
    args = SFTConfig(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ,
        average_tokens_across_devices = False,
        remove_unused_columns = False,
        per_device_train_batch_size = 2,      # L4: keep 2; A100: can raise to 4
        gradient_accumulation_steps = 8,      # effective batch 16
        warmup_ratio = 0.03,
        num_train_epochs = 2,
        learning_rate = 1e-4,                 # voice pass on a fresh r=32 adapter
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/content/outputs",
        report_to = "none",
    ),
)

# Gemma 4 turn markers — not the previous architecture's, and not Llama's header ids.
trainer = train_on_responses_only(
    trainer,
    instruction_part = INSTRUCTION_PART,
    response_part    = RESPONSE_PART,
)
print("trainer ready — masking on", repr(RESPONSE_PART), "..", repr(EOT_TOKEN))


In [ ]:
# ======================= CHECK 2 — response masking (THE critical gate) =======================
# Wrong offsets here silently train on the prompt. The Gemma-3 version of this check
# only asserted "something was unmasked", which a PARTIAL mis-mask passes: leak the
# first few tokens of the reply into the trained span, mask the rest, and it still
# passes. So the trained text is compared against the row's own final assistant
# content — both its start and its full length.
import torch

def decode_pair(i):
    ex = trainer.train_dataset[i]
    ids, labels = ex["input_ids"], ex["labels"]
    kept   = [t for t, l in zip(ids, labels) if l != -100]
    masked = [t for t, l in zip(ids, labels) if l == -100]
    return tok.decode(masked), tok.decode(kept)

ok = True
for i in [0, 1, 2, len(trainer.train_dataset) // 2, len(trainer.train_dataset) - 1]:
    masked_txt, kept_txt = decode_pair(i)
    cond_a = "<|turn>user" in masked_txt           # the prompt turn is masked out
    cond_b = "<|turn>user" not in kept_txt         # ... and is not in the trained span
    cond_c = len(kept_txt.strip()) > 0             # something IS trained
    print(f"[{i}] prompt_masked={cond_a} trained_clean={cond_b} nonempty={cond_c}")
    print("    TRAINED:", repr(kept_txt[:120]))
    ok &= (cond_a and cond_b and cond_c)

assert ok, ("MASKING BROKEN — do not train. Check instruction_part / response_part: "
            f"expected {INSTRUCTION_PART!r} and {RESPONSE_PART!r}.")

# --- the strengthened check: example 0's trained text IS that row's final reply ---
expected0 = last_assistant(examples[0]["messages"]).strip()
_, kept0  = decode_pair(0)
trained0  = norm(kept0)
print("\nexample 0 expected tail:", repr(expected0[:160]))
print("example 0 trained text :", repr(trained0[:160]))
assert trained0.startswith(norm(expected0)), (
    "PARTIAL MIS-MASK — the unmasked text does not start with example 0's final "
    f"assistant content.\n  expected to start with: {norm(expected0)[:120]!r}\n"
    f"  actually starts with:  {trained0[:120]!r}\n"
    "A merely non-empty mask passes the old check; this one does not. Do not train.")
assert norm(expected0) in trained0, (
    "PARTIAL MIS-MASK — the trained span starts correctly but is truncated; the tail of "
    "example 0's reply is never trained on.")
assert not any(p.strip() for p in thought_payloads(kept0)), \
    "a thought channel ended up inside the trained span"
assert_no_thoughts(texts, "corpus rows")

frac = float(sum((torch.tensor(trainer.train_dataset[i]['labels']) != -100).float().mean()
                 for i in range(50)) / 50)
print(f"\nmean trained-token fraction: {frac:.2f}  (expect ~0.2-0.6)")
assert 0.05 < frac < 0.9, "suspicious trained fraction"
print("CHECK 2 PASSED — training on model turns only, content verified")


In [ ]:
# --- token_type_ids: PROBE, do not assume -----------------------------------
# Gemma 3's causal-mask builder needed a zero-filled token_type_ids on every batch.
# That was architecture-specific. Do not delete it on faith and do not keep it on
# faith either: push one real batch through the model and let the error decide.
_base_collator = trainer.data_collator

def _collator(features):
    return _base_collator(features)

def _one_real_batch(collator):
    batch = collator([trainer.train_dataset[i] for i in range(2)])
    batch = {k: (v.to("cuda") if hasattr(v, "to") else v) for k, v in batch.items()}
    with torch.no_grad():
        model(**batch)
    return batch

trainer.data_collator = _collator
try:
    batch = _one_real_batch(_collator)
    TOKEN_TYPE_IDS_PATH = "NOT NEEDED — the model accepted a real batch without token_type_ids"
except Exception as e:
    if "token_type_ids" not in str(e):
        raise
    def _collator_with_token_type_ids(features):
        b = _collator(features)
        if "token_type_ids" not in b:
            b["token_type_ids"] = torch.zeros_like(b["input_ids"])
        return b
    trainer.data_collator = _collator_with_token_type_ids
    batch = _one_real_batch(_collator_with_token_type_ids)
    TOKEN_TYPE_IDS_PATH = ("REQUIRED — the error named token_type_ids, so the Gemma-3 "
                           "zero-fill wrapper is installed")

print("token_type_ids path:", TOKEN_TYPE_IDS_PATH)
print("  batch keys:", sorted(batch.keys()))
assert "labels" in batch and (batch["labels"] == -100).any(), \
    "response masking must still be present in a real batch"
if "token_type_ids" in batch:
    assert batch["token_type_ids"].shape == batch["input_ids"].shape
    assert (batch["token_type_ids"] == 0).all(), "text-only training expects all zeros"
    print("  token_type_ids:", tuple(batch["token_type_ids"].shape),
          "| masked tokens in batch:", int((batch["labels"] == -100).sum()))
print("collator ready")


In [ ]:
# 8) Train
stats = trainer.train()
print(stats)
print(f"\nmean train runtime {stats.metrics.get('train_runtime', 0)/60:.1f} min "
      f"({stats.metrics.get('train_samples_per_second', 0):.2f} samples/s)")


In [ ]:
# 9) Persist the adapter BEFORE the behavioural gate — Phase 2 needs this folder
model.save_pretrained(OUT_ADAPTER)
tokenizer.save_pretrained(OUT_ADAPTER)
import os
print("saved:", OUT_ADAPTER)
for f in sorted(os.listdir(OUT_ADAPTER)):
    print("  ", f, os.path.getsize(os.path.join(OUT_ADAPTER, f)))
assert os.path.exists(os.path.join(OUT_ADAPTER, "adapter_model.safetensors")), \
    "adapter weights missing!"
print("\nSaved. Phase 2 merges this exact folder — keep the name as it is.")


In [ ]:
# ============== CHECK 3 — behavioural smoke test (runs after the save) ==============
from transformers import TextStreamer
FastModel.for_inference(model)

def say(prompt, max_new_tokens=180):
    # CORE and the identity card live in the RAG layer at runtime, so this probe
    # sends the bare user turn — and renders with thinking disabled.
    text = render([{"role": "user", "content": prompt}], add_generation_prompt=True)
    inp  = tok(text, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**inp, max_new_tokens=max_new_tokens,
                         eos_token_id=[tokenizer.eos_token_id, EOT_ID],
                         pad_token_id=tokenizer.eos_token_id,
                         temperature=0.7, top_p=0.95, do_sample=True)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True).strip()

tests = ["hey",
         "do you have a cat",
         "what do you think of the internet?",
         'Analyze this line: "The sea returned everything except the swimmer."']
for t in tests:
    print("="*70); print("USER:", t); print("CAMUS:", say(t))

print("\n" + "="*70)
print("""MANUAL GATE — read the four answers above:
  [ ] first person throughout (no 'Camus would...', no third person)
  [ ] no stage directions in parentheses (the roleplay tic)
  [ ] not every answer ends with a question back (the closer tic)
  [ ] greeting is SHORT; analysis engages instead of disowning
  [ ] no thought/reasoning text leaking into the reply
If two or more fail, stop and adjust data/epochs rather than proceeding.""")


In [ ]:
# 10) Wall clock
print(f"\nwall-clock elapsed for Phase 1 ({PHASE1_CORPUS}): {(time.time() - T0)/60:.1f} min")


## Record the elapsed time

The cell above prints the wall-clock figure; the same number is available as
`(time.time() - T0)/60` at any point after it.

**Please write that number into `docs/ROADMAP.md`.** `docs/gemma4_assessment.md` §6 could
only give a 2.5–4 h *planning range* for the session, because no Gemma-3 run time is
recorded anywhere in this repository — no notebook output, no ROADMAP entry, no log. That
gap is the entire reason that section is a range. Record the whole session, not just
`train_runtime`: the Drive mount, the export and the upload are part of what a re-run
costs, and if you are running both corpora, record the two figures separately.
